# Análisis de perfil y desempeño 08/06/2026

Influeixen l’antiguitat, la càrrega laboral diària o la distància fins al lloc de treball en el rendiment dels nostres col·laboradors?
Podem identificar perfils amb major risc de baix rendiment i fer ajustaments en l’organització per potenciar els resultats?

Mensaje de Verónica:
En vuestro caso hay que dosificar más el esfuerzo y la complejidad. He contado 6 técnicas creo para una pregunta que a nivel general es sencilla. Recomiendo ir por descriptivo+ una técnica más (o bien RLM si el score lo trabajan como variable continua o regresión logística multinomial si la trabajan como categórica). En vuestro caso, diría que el "error" está en intentar deducir de cada parte de la pregunta una técnica de análisis. 

# Librerías

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import datetime as dt
from datetime import date
import scipy.stats as stats
import numpy as np
import plotly.graph_objs as go
import seaborn as sns
from scipy.stats import spearmanr 
from scipy.stats import mannwhitneyu
import statsmodels.formula.api as smf
from matplotlib.lines import Line2D
from sklearn.preprocessing import StandardScaler


# Funciones

In [2]:
def distr_hist(df: pd.DataFrame, column: str, bars: int) -> None:
    """
    Muestra un histograma para una columna del DataFrame.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame con los datos.
    column : str
        Columna a visualizar.
    bars : int
        Número de barras del histograma.
    """
    plt.hist(df[column], bins=bars)
    plt.title(f'Distribución de la variable {column}')
    plt.show()

In [3]:
def distr_value(df: pd.DataFrame, column: str) -> None:
    """
    Muestra la distribución porcentual y los valores únicos de una columna.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame con los datos.
    column : str
        Columna a analizar.
    """
    value_count = df[column].value_counts(normalize=True)*100
    print(f'Value counts: {value_count}')
    val_unique = df[column].unique()
    print(f'----\nValores únicos:{val_unique}')
    
    print('----\nGráfico distribución valores únicos:')
    (df[column]
    .value_counts(normalize=True)
    .mul(100)
    .plot(kind='bar'))

    plt.ylabel('Porcentaje (%)')
    plt.xlabel('Hit_target')
    plt.title('Distribución de Hit_target')
    plt.show()                  

In [4]:
def distr_boxplot(df: pd.DataFrame, column: str) -> None:
    """
    Muestra un boxplot para una columna del DataFrame.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame con los datos.
    column : str
        Columna a visualizar.
    """
    plt.boxplot(df[column])
    plt.title(f'Boxplot de la variable {column}')
    plt.show()

In [5]:
def outliers_column(df: pd.DataFrame, column: str) -> pd.DataFrame:
    """
    Devuelve las filas con valores atípicos en una columna usando el método IQR.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame con los datos.
    column : str
        Columna numérica a analizar.
    """
    col = df[column]

    Q1 = col.quantile(0.25)
    Q3 = col.quantile(0.75)
    IQR = Q3 - Q1

    limit_inferior = Q1 - 1.5 * IQR
    limit_superior = Q3 + 1.5 * IQR

    outliers = df[(col < limit_inferior) | (col > limit_superior)]
    
    return outliers

# Documento

In [6]:
df_RRHH = pd.read_csv('../../Data/clean/clean_data_08062026.csv')
df_RRHH

,ID,Reason_absence,Reason_absence_name,Month_absence,Month_absence_name,Day_week,Day_week_name,Seasons,Seasons_name,Transportation_expense,...,Son,Social_drinker,Social_smoker,Pet,Weight,Height,Body_mass_index,BMI_calculated,Absenteeism_hours,importacion
0,14,11.0,Sistema digestivo,5.0,Mayo,2,Lunes,2,Primavera,155,...,2,1,0,0,95,196,25,24.7,120,2026-05-25
1,36,13.0,Sistema musculoesquelético,10.0,Octubre,4,Miércoles,4,Otoño,118,...,1,1,0,0,98,178,31,30.9,120,2026-05-25
2,9,6.0,Sistema nervioso,1.0,Enero,3,Martes,1,Invierno,228,...,2,0,0,1,65,172,22,22.0,120,2026-05-25
3,28,9.0,Sistema circulatorio,1.0,Enero,3,Martes,1,Invierno,225,...,1,0,0,2,69,169,24,24.2,112,2026-05-25
4,9,12.0,Piel y tejido subcutáneo,9.0,Septiembre,3,Martes,3,Verano,228,...,2,0,0,1,65,172,22,22.0,112,2026-05-25
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1104,22,1.0,Enfermedades infecciosas y parasitarias,4.0,Abril,4,Miércoles,2,Primavera,179,...,0,0,0,0,56,171,19,19.2,64,2026-06-08
1105,26,19.0,"Lesiones, intoxicaciones y consecuencias externas",11.0,Noviembre,6,Viernes,4,Otoño,300,...,2,1,1,1,77,175,25,25.1,64,2026-06-08
1106,34,19.0,"Lesiones, intoxicaciones y consecuencias externas",6.0,Junio,3,Martes,2,Primavera,118,...,0,0,0,0,83,172,28,28.1,56,2026-06-08
1107,20,19.0,"Lesiones, intoxicaciones y consecuencias externas",10.0,Octubre,6,Viernes,4,Otoño,260,...,4,1,0,0,65,168,23,23.0,56,2026-06-08


In [7]:
df_RRHH.info()

<class 'pandas.DataFrame'>
RangeIndex: 1109 entries, 0 to 1108
Data columns (total 28 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   ID                       1109 non-null   int64  
 1   Reason_absence           1042 non-null   float64
 2   Reason_absence_name      1042 non-null   str    
 3   Month_absence            1106 non-null   float64
 4   Month_absence_name       1106 non-null   str    
 5   Day_week                 1109 non-null   int64  
 6   Day_week_name            1109 non-null   str    
 7   Seasons                  1109 non-null   int64  
 8   Seasons_name             1109 non-null   str    
 9   Transportation_expense   1109 non-null   int64  
 10  Distance_Residence_Work  1109 non-null   int64  
 11  Service_time             1109 non-null   int64  
 12  Age                      1109 non-null   int64  
 13  Work_load_Average_day    1109 non-null   float64
 14  Hit_target               1109 non-n

In [8]:
df_RRHH.shape

(1109, 28)

In [10]:
df_RRHH.describe().T

,count,mean,std,min,25%,50%,75%,max
ID,1109.0,79.120830,106.751321,1.000,13.000,28.000,109.000,386.000
Reason_absence,1042.0,20.215931,7.179088,1.000,13.000,23.000,26.000,28.000
Month_absence,1106.0,6.472875,3.449400,1.000,4.000,7.000,9.000,12.000
Day_week,1109.0,3.904418,1.433819,2.000,3.000,4.000,5.000,6.000
Seasons,1109.0,2.516682,1.108281,1.000,2.000,3.000,4.000,4.000
Transportation_expense,1109.0,221.312894,66.615474,118.000,179.000,225.000,260.000,388.000
Distance_Residence_Work,1109.0,29.507665,14.804538,5.000,16.000,26.000,50.000,52.000
Service_time,1109.0,12.655546,4.369696,1.000,9.000,13.000,16.000,29.000
Age,1109.0,36.647430,6.655875,27.000,31.000,37.000,40.000,58.000
Work_load_Average_day,1109.0,270.145783,38.987515,205.917,241.476,264.249,284.853,378.884


In [11]:
df_RRHH.columns

Index(['ID', 'Reason_absence', 'Reason_absence_name', 'Month_absence',
       'Month_absence_name', 'Day_week', 'Day_week_name', 'Seasons',
       'Seasons_name', 'Transportation_expense', 'Distance_Residence_Work',
       'Service_time', 'Age', 'Work_load_Average_day', 'Hit_target',
       'Disciplinary_failure', 'Education', 'Education_name', 'Son',
       'Social_drinker', 'Social_smoker', 'Pet', 'Weight', 'Height',
       'Body_mass_index', 'BMI_calculated', 'Absenteeism_hours',
       'importacion'],
      dtype='str')

Listas de columnas numéricas y categóricas separadas de todo el dataset (con 'ID')

In [12]:
num = ['ID','Transportation_expense', 'Distance_Residence_Work', 'Service_time',
       'Age', 'Work_load_Average_day', 'Hit_target','Pet', 'Son',
       'Weight', 'Height', 'Absenteeism_hours', 'BMI_calculated']

cat = ['ID','Reason_absence', 'Month_absence', 'Day_week',
       'Seasons', 'Disciplinary_failure', 'Education',
       'Social_drinker', 'Social_smoker']

## Tamaño de muestra

In [13]:
df_RRHH.info()

<class 'pandas.DataFrame'>
RangeIndex: 1109 entries, 0 to 1108
Data columns (total 28 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   ID                       1109 non-null   int64  
 1   Reason_absence           1042 non-null   float64
 2   Reason_absence_name      1042 non-null   str    
 3   Month_absence            1106 non-null   float64
 4   Month_absence_name       1106 non-null   str    
 5   Day_week                 1109 non-null   int64  
 6   Day_week_name            1109 non-null   str    
 7   Seasons                  1109 non-null   int64  
 8   Seasons_name             1109 non-null   str    
 9   Transportation_expense   1109 non-null   int64  
 10  Distance_Residence_Work  1109 non-null   int64  
 11  Service_time             1109 non-null   int64  
 12  Age                      1109 non-null   int64  
 13  Work_load_Average_day    1109 non-null   float64
 14  Hit_target               1109 non-n

In [14]:
# Función para calcular la moda de variables categóricas/temporales
def moda(x):
    m = x.dropna().mode()
    return m.iloc[0] if len(m) > 0 else np.nan

df_emp = df_RRHH.groupby('ID').agg(
    # Variable dependiente agregada a nivel empleado
    Hit_target_median=('Hit_target', 'median'),
    Hit_target_mean=('Hit_target', 'mean'),

    # Número de solicitudes / registros de ausencia por empleado
    num_absences=('ID', 'size'),

    # Variables numéricas del empleado
    Age=('Age', 'first'),
    Service_time=('Service_time', 'first'),
    Distance_Residence_Work=('Distance_Residence_Work', 'first'),
    Transportation_expense=('Transportation_expense', 'first'),

    # Variables que pueden variar por registro
    Absenteeism_hours_total=('Absenteeism_hours', 'sum'),
    Absenteeism_hours_median=('Absenteeism_hours', 'median'),
    # Work_load_Average_day_median=('Work_load_Average_day', 'median'),

    # Variables categóricas/dummy del empleado
    Disciplinary_failure=('Disciplinary_failure', 'first'),
    Education=('Education', 'first'),
    Son=('Son', 'first'),
    Social_drinker=('Social_drinker', 'first'),
    Social_smoker=('Social_smoker', 'first'),
    Pet=('Pet', 'first'),

    # Variables temporales: valor más frecuente por empleado
    Month_absence_mode=('Month_absence', moda),
    Seasons_mode=('Seasons', moda)
).reset_index()

In [15]:
variables_grupo = [
    'Disciplinary_failure',
    'Education',
    'Son',
    'Social_drinker',
    'Social_smoker',
    'Pet'
]

for var in variables_grupo:
    #print(f"\n{var}")
    print(df_emp[var].value_counts(dropna=False))
    print(df_emp[var].value_counts(normalize=True, dropna=False) * 100)

Disciplinary_failure
0    360
1     26
Name: count, dtype: int64
Disciplinary_failure
0    93.264249
1     6.735751
Name: proportion, dtype: float64
Education
1    321
3     31
2     31
4      3
Name: count, dtype: int64
Education
1    83.160622
3     8.031088
2     8.031088
4     0.777202
Name: proportion, dtype: float64
Son
0    140
1    114
2    102
4     23
3      7
Name: count, dtype: int64
Son
0    36.269430
1    29.533679
2    26.424870
4     5.958549
3     1.813472
Name: proportion, dtype: float64
Social_drinker
1    218
0    168
Name: count, dtype: int64
Social_drinker
1    56.476684
0    43.523316
Name: proportion, dtype: float64
Social_smoker
0    340
1     46
Name: count, dtype: int64
Social_smoker
0    88.082902
1    11.917098
Name: proportion, dtype: float64
Pet
0    226
1     80
2     50
4     17
5      8
8      5
Name: count, dtype: int64
Pet
0    58.549223
1    20.725389
2    12.953368
4     4.404145
5     2.072539
8     1.295337
Name: proportion, dtype: float64


# Pregunta de negocio

**Analista de Perfil i Desempeño Professional:**  
- Influeixen l’antiguitat, la càrrega laboral diària o la distància fins al lloc de treball en el rendiment dels nostres col·laboradors?
- Podem identificar perfils amb major risc de baix rendiment i fer ajustaments en l’organització per potenciar els resultats?

- [KPIs para empresas de transporte: 10 indicadores clave](https://www.lextransport.es/kpis-para-empresas-de-transporte/)
- [Descubre qué es el rendimiento laboral y su importancia](https://mx.indeed.com/orientacion-profesional/desarrollo-profesional/rendimiento-laboral)

- [8 estrategias para aumentar el rendimiento de tu flota](https://www.eurowag.com/es/blog/8-estrategias-para-aumentar-el-rendimiento-de-tu-flota)

## Crear score **Rendimiento**

### Subset de las variables a estudiar  
Variables influencia:
- Antigüedad (Service_time)
- Carga Trabajo
- Distancia al trabajo

Variables score rendimiento:
- Horas absentismo
- Desempeño (Hit target)
- Disciplinary failure

Variables contexto:
- ID
- Mes (+ nombre)
- fecha importacion

In [16]:
df_RRHH.columns

Index(['ID', 'Reason_absence', 'Reason_absence_name', 'Month_absence',
       'Month_absence_name', 'Day_week', 'Day_week_name', 'Seasons',
       'Seasons_name', 'Transportation_expense', 'Distance_Residence_Work',
       'Service_time', 'Age', 'Work_load_Average_day', 'Hit_target',
       'Disciplinary_failure', 'Education', 'Education_name', 'Son',
       'Social_drinker', 'Social_smoker', 'Pet', 'Weight', 'Height',
       'Body_mass_index', 'BMI_calculated', 'Absenteeism_hours',
       'importacion'],
      dtype='str')

In [33]:
var_subset = ['ID', 'Month_absence','Month_absence_name',
              'Distance_Residence_Work','Service_time', 'Work_load_Average_day',
              'Hit_target','Disciplinary_failure','Absenteeism_hours']

In [34]:
df_RRHH_subset = df_RRHH[var_subset].copy()
df_RRHH_subset

,ID,Month_absence,Month_absence_name,Distance_Residence_Work,Service_time,Work_load_Average_day,Hit_target,Disciplinary_failure,Absenteeism_hours
0,14,5.0,Mayo,12,14,284.031,97,0,120
1,36,10.0,Octubre,13,18,239.409,98,0,120
2,9,1.0,Enero,14,16,264.604,93,0,120
3,28,1.0,Enero,26,9,230.290,92,0,112
4,9,9.0,Septiembre,14,16,222.196,99,0,112
...,...,...,...,...,...,...,...,...,...
1104,22,4.0,Abril,26,9,265.017,88,0,64
1105,26,11.0,Noviembre,26,13,237.656,99,0,64
1106,34,6.0,Junio,10,10,261.306,97,0,56
1107,20,10.0,Octubre,50,11,326.452,96,0,56


### Agrupación por ID y mes

In [35]:
df_grouped_ID_month = (
    df_RRHH_subset
    .groupby(['ID', 'Month_absence'], as_index=False)
    .agg({
        'Month_absence_name': 'first',
        'Distance_Residence_Work': 'first',
        'Service_time': 'first',
        'Work_load_Average_day': 'median',
        'Hit_target': 'first',
        'Disciplinary_failure': 'max',
        'Absenteeism_hours': 'sum',
    })
)

df_grouped_ID_month['Work_load_Average_day'] = df_grouped_ID_month['Work_load_Average_day'].round(3)

In [36]:
df_grouped_ID_month

,ID,Month_absence,Month_absence_name,Distance_Residence_Work,Service_time,Work_load_Average_day,Hit_target,Disciplinary_failure,Absenteeism_hours
0,1,1.0,Enero,11,14,252.079,97,0,12
1,1,2.0,Febrero,11,14,257.706,94,0,17
2,1,4.0,Abril,11,14,265.017,88,0,4
3,1,5.0,Mayo,11,14,284.031,97,0,3
4,1,6.0,Junio,11,14,236.629,97,0,17
...,...,...,...,...,...,...,...,...,...
579,382,1.0,Enero,36,13,275.312,98,0,32
580,383,8.0,Agosto,51,18,251.818,96,0,3
581,384,1.0,Enero,36,13,275.312,98,0,8
582,385,8.0,Agosto,27,6,251.818,96,0,16


In [37]:
df_grouped_ID_month.info()

<class 'pandas.DataFrame'>
RangeIndex: 584 entries, 0 to 583
Data columns (total 9 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   ID                       584 non-null    int64  
 1   Month_absence            584 non-null    float64
 2   Month_absence_name       584 non-null    str    
 3   Distance_Residence_Work  584 non-null    int64  
 4   Service_time             584 non-null    int64  
 5   Work_load_Average_day    584 non-null    float64
 6   Hit_target               584 non-null    int64  
 7   Disciplinary_failure     584 non-null    int64  
 8   Absenteeism_hours        584 non-null    int64  
dtypes: float64(2), int64(6), str(1)
memory usage: 41.2 KB


### Cálculos score

Capacity

In [ ]:
A_max = df_grouped_ID_month['Absenteeism_hours'].max()

df_grouped_ID_month['Capacity'] = 1 - df_grouped_ID_month['Absenteeism_hours'] / A_max

# Si quieres evitar valores negativos por algún outlier raro:
#df['Capacity'] = df['Capacity'].clip(lower=0)

Disciplinary failure, factor delta pequeño (0.10, 10% de penalización si hay disciplinary_failure)
- Si Disciplinary_failure es 0 -> 1 (sin penalización)
- Si es 1 -> 0.90 (10% de penalización)

In [39]:
delta = 0.10

df_grouped_ID_month['Disciplinary_component'] = 1 - delta * df_grouped_ID_month['Disciplinary_failure']

Hit_target normalizado por todos los meses, HitNorm estará entre 0 y 1

In [40]:
H_min = df_grouped_ID_month['Hit_target'].min()
H_max = df_grouped_ID_month['Hit_target'].max()

df_grouped_ID_month['HitNorm'] = (df_grouped_ID_month['Hit_target'] - H_min) / (H_max - H_min)

**Score final de rendimiento**  
Elegimos gamma para que Hit_target tenga más peso que disciplinary (0.30)

In [43]:
gamma = 0.30

df_grouped_ID_month['Score_rendimiento'] = (
    df_grouped_ID_month['Capacity'] *
    df_grouped_ID_month['Disciplinary_component'] *
    (1 + gamma * df_grouped_ID_month['HitNorm'])
)

Score final de rendimiento sin normalizar:
- Descriptivo.
- Crear categorías (bajo/medio/alto) con cuartiles.
- Regresión logística (la variable es categórica, no el valor numérico).

*Normalizar* el score entre 0 y 1:
- Gráficos más limpios (0–1).
- Tablas comparativas donde quieres que el rango sea estándar.
- Presentación visual (no afecta los análisis).

In [44]:
S_min = df_grouped_ID_month['Score_rendimiento'].min()
S_max = df_grouped_ID_month['Score_rendimiento'].max()

df_grouped_ID_month['Score_rendimiento_norm'] = (df_grouped_ID_month['Score_rendimiento'] - S_min) / (S_max - S_min)

In [45]:
df_grouped_ID_month

,ID,Month_absence,Month_absence_name,Distance_Residence_Work,Service_time,Work_load_Average_day,Hit_target,Disciplinary_failure,Absenteeism_hours,Capacity,Disciplinary_component,HitNorm,Score_rendimiento,Score_rendimiento_norm
0,1,1.0,Enero,11,14,252.079,97,0,12,0.9700,1.0,0.842105,1.215053,0.936998
1,1,2.0,Febrero,11,14,257.706,94,0,17,0.9575,1.0,0.684211,1.154039,0.889948
2,1,4.0,Abril,11,14,265.017,88,0,4,0.9900,1.0,0.368421,1.099421,0.847828
3,1,5.0,Mayo,11,14,284.031,97,0,3,0.9925,1.0,0.842105,1.243237,0.958733
4,1,6.0,Junio,11,14,236.629,97,0,17,0.9575,1.0,0.842105,1.199395,0.924924
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
579,382,1.0,Enero,36,13,275.312,98,0,32,0.9200,1.0,0.894737,1.166947,0.899902
580,383,8.0,Agosto,51,18,251.818,96,0,3,0.9925,1.0,0.789474,1.227566,0.946648
581,384,1.0,Enero,36,13,275.312,98,0,8,0.9800,1.0,0.894737,1.243053,0.958591
582,385,8.0,Agosto,27,6,251.818,96,0,16,0.9600,1.0,0.789474,1.187368,0.915649


In [46]:
df_grouped_ID_month.describe().T

,count,mean,std,min,25%,50%,75%,max
ID,584.0,134.082192,123.089280,1.000,22.000000,94.500000,240.250000,386.00000
Month_absence,584.0,6.417808,3.514258,1.000,3.000000,6.000000,9.000000,12.00000
Distance_Residence_Work,584.0,28.837329,14.535905,5.000,16.000000,26.000000,42.000000,52.00000
Service_time,584.0,12.505137,4.519442,1.000,9.000000,13.000000,16.000000,29.00000
Work_load_Average_day,584.0,269.376868,38.470980,205.917,241.476000,264.249000,284.853000,378.88400
Hit_target,584.0,94.599315,3.764108,81.000,92.750000,95.000000,97.000000,100.00000
Disciplinary_failure,584.0,0.102740,0.303879,0.000,0.000000,0.000000,0.000000,1.00000
Absenteeism_hours,584.0,17.388699,42.726638,0.000,2.000000,8.000000,12.000000,400.00000
Capacity,584.0,0.956528,0.106817,0.000,0.970000,0.980000,0.995000,1.00000
Disciplinary_component,584.0,0.989726,0.030388,0.900,1.000000,1.000000,1.000000,1.00000


Tu máximo de 1.29 es exactamente lo esperado: alguien con:
- Capacity ≈ 1 (casi sin absentismo),
- Sin disciplinary_failure,
- HitNorm ≈ 1 (mejor mes),
- obtendría un score cercano a 1.30.

## Subset agrupado por ID con rendimiento para los análisis

In [48]:
df_rendimiento_grouped_ID = df_grouped_ID_month.groupby('ID').agg({
        'Distance_Residence_Work': 'first',
        'Service_time': 'first',
        'Work_load_Average_day': 'median',
        'Hit_target': 'first',
        'Disciplinary_failure': 'max',
        'Absenteeism_hours': 'sum',
        'Score_rendimiento': 'median'
    }).reset_index()

In [49]:
df_rendimiento_grouped_ID

,ID,Distance_Residence_Work,Service_time,Work_load_Average_day,Hit_target,Disciplinary_failure,Absenteeism_hours,Score_rendimiento
0,1,11,14,261.3615,97,1,121,1.176717
1,2,29,12,252.6895,92,1,25,1.113158
2,3,51,18,253.7110,97,1,482,1.114145
3,5,20,13,258.5890,92,1,104,1.141895
4,6,29,13,284.8530,93,0,72,1.134737
...,...,...,...,...,...,...,...,...
379,382,36,13,275.3120,98,0,32,1.166947
380,383,51,18,251.8180,96,0,3,1.227566
381,384,36,13,275.3120,98,0,8,1.243053
382,385,27,6,251.8180,96,0,16,1.187368


A partir de aquí no actualizado con datos actuales

## Análisis descriptivo cambiar para las variables de análisis

### Distribución variable score_rendimiento (histograma y boxplot [outliers])

In [ ]:
distr_hist(df_RRHH_ind_grouped,'Hit_target',4)

In [ ]:
distr_value(df_RRHH_ind_grouped,'Hit_target')

In [ ]:
distr_boxplot(df_RRHH_ind_grouped,'Hit_target')

Para saber el valor del outlier he mirado qué valores quedan fuera del cuartil Q1 y Q3.  
El valor **outlier es 81** (hay 23 registros), que es el mínimo, el siguiente valor es 88.

In [ ]:
outliers_hit_target_all = outliers_column(df_RRHH,'Hit_target')
outliers_hit_target_all

Distribución agrupada por individuo

In [ ]:
distr_hist(df_RRHH_ind_grouped,'Hit_target',4)

In [ ]:
distr_value(df_RRHH_ind_grouped,'Hit_target')

In [ ]:
distr_boxplot(df_RRHH_ind_grouped,'Hit_target')

Para saber el valor del outlier he mirado qué valores quedan fuera del cuartil Q1 y Q3.  
El valor **outlier es 81** (hay 23 registros), que es el mínimo, el siguiente valor es 88.

In [ ]:
outliers_hit_target_all = outliers_column(df_RRHH,'Hit_target')
outliers_hit_target_all

## Correlacion Spearman

Se utiliza Spearman porque Hit_target no es normal

In [ ]:
spearman_corr = df_RRHH_ind_grouped[num_ID].corr(method='spearman')

#Extracción específica de asociaciones con Hit_target (Desempeño)
target_corr = spearman_corr['Hit_target'].sort_values(ascending=False)
print("Asociaciones con el Desempeño (Hit_target):")
print(target_corr)

### Visualización Spearman

In [ ]:
correlaciones_target = pd.Series({
    'Height': 0.104287,
    'Absence_count': 0.049381,
    'Absenteeism_hours': 0.015056,
    'Distance_Residence_Work': -0.001216,
    'Son': -0.031321,
    'Service_time': -0.067977,
    'Pet': -0.077616,
    'Age': -0.080973,
    'Work_load_Average_day': -0.103270,
    'Transportation_expense': -0.109586,
    'Weight': -0.139090,
    'BMI_calculated': -0.195817
}).sort_values(ascending=False)

# 2. Configuración de la visualización
plt.figure(figsize=(10, 6), facecolor='none')

colors = ['#2ecc71' if x > 0 else '#e74c3c' for x in correlaciones_target.values]

ax= sns.barplot(x=correlaciones_target.values, y=correlaciones_target.index, palette=colors)

ax.set_facecolor('none')

# 3. Estética y etiquetas
plt.title('Asociación de Variables con el Desempeño (Spearman rho)', fontsize=14)
plt.xlabel('Coeficiente de Correlación')
plt.ylabel('Características del Perfil')
plt.axvline(0, color='black', linestyle='-', linewidth=1) # Línea central
plt.grid(axis='x', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

In [ ]:
#Definimos las variables numéricas del perfil (excluyendo ID y el propio Hit_target)
variables_analisis = [
    'BMI_calculated', 'Weight', 'Transportation_expense', 
    'Work_load_Average_day', 'Age', 'Pet', 'Service_time', 
    'Son', 'Height', 'Distance_Residence_Work', 
    'Absence_count', 'Absenteeism_hours'
]

#2.Iteramos para calcular Coeficiente y P-Value
resultados_stats = []

for var in variables_analisis:

    df_temp = df_RRHH_ind_grouped[[var, 'Hit_target']].dropna()
    
    coef, p_val = spearmanr(df_temp[var], df_temp['Hit_target'])
    
    significativo = "SÍ" if p_val < 0.05 else "NO"
    
    resultados_stats.append({
        'Variable': var,
        'Spearman_Rho': round(coef, 4),
        'P-Value': round(p_val, 4),
        'Significativo (p < 0.05)': significativo
    })

df_validacion = pd.DataFrame(resultados_stats).sort_values(by='P-Value')

In [ ]:
print(df_validacion)

## Mann-Withney U

In [ ]:
def analisis_mw_boxplot(df, variables_binarias, target='Hit_target'):
    """
    Realiza el test de Mann-Whitney U y genera visualizaciones de tipo Boxplot
    para comparar el desempeño entre grupos binarios.
    """
    resultados_mw = []

    fig, axes = plt.subplots(
        1,
        len(variables_binarias),
        figsize=(6 * len(variables_binarias), 6),
        sharey=True,
        facecolor='none'
    )

    if len(variables_binarias) == 1:
        axes = [axes]

    for i, var in enumerate(variables_binarias):

        grupo_0 = df[df[var] == 0][target].dropna()
        grupo_1 = df[df[var] == 1][target].dropna()

        stat, p_val = mannwhitneyu(
            grupo_0,
            grupo_1,
            alternative='two-sided'
        )

        significativo = "SÍ" if p_val < 0.05 else "NO"

        resultados_mw.append({
            'Variable': var,
            'Estadístico U': stat,
            'P-Value': round(p_val, 4),
            'Diferencia Significativa': significativo
        })

        ax = axes[i]

        sns.boxplot(
            ax=ax,
            x=var,
            y=target,
            data=df,
            palette="Set2"
        )

        ax.set_facecolor('none')

        ax.set_title(f'Comparación de {target}\npor {var}')
        ax.set_xlabel(f'{var} (0=No, 1=Sí)')
        ax.set_ylabel(target if i == 0 else "")

    plt.tight_layout()
    plt.show()

    return pd.DataFrame(resultados_mw)

In [ ]:
vars_estudio = ['Social_drinker', 'Social_smoker', 'Disciplinary_failure']
df_stats_boxplot = analisis_mw_boxplot(df_RRHH_ind_grouped, vars_estudio)
print(df_stats_boxplot)

No se encontraron evidencias de que el consumo de alcohol, el tabaquismo o los fallos disciplinarios estén relacionados con diferencias en el desempeño laboral. No obstante, la baja variabilidad de Hit_target y el reducido número de empleados con incidencias disciplinarias limitan la capacidad para detectar posibles efectos.


##### Disciplinary failure

In [ ]:
df_RRHH_ind_grouped['Disciplinary_failure'].value_counts()

In [ ]:
group_0 = df_RRHH_ind_grouped.loc[
    df_RRHH_ind_grouped['Disciplinary_failure'] == 0,
    'Hit_target'
].dropna()

group_1 = df_RRHH_ind_grouped.loc[
    df_RRHH_ind_grouped['Disciplinary_failure'] == 1,
    'Hit_target'
].dropna()

u_statistic, p_value = stats.mannwhitneyu(
    group_0,
    group_1,
    alternative='two-sided'
)

print(f'U statistic: {u_statistic}')
print(f'p-value: {p_value}')

if p_value < 0.05:
    print('Reject H0: hay diferencias estadísticamente significativas entre los grupos.')
else:
    print('Fail to reject H0: no hay diferencias estadísticamente significativas entre los grupos.')

In [ ]:
group_0.describe().T

In [ ]:
group_1.describe().T

#### Variables a categorizar

##### Absence count

In [ ]:
df_RRHH_ind_grouped['Absence_count'].value_counts(normalize=True).sort_index()

In [ ]:
df_abs_count_hit_target = df_RRHH_ind_grouped[['ID','Absence_count','Hit_target']].copy()
df_abs_count_hit_target

In [ ]:
df_abs_count_hit_target['category'] = np.where(
    df_abs_count_hit_target['Absence_count'] == 1,
    '1 ausencia',
    '>1 ausencias'
)

In [ ]:
df_abs_count_hit_target['category'].value_counts()

In [ ]:
group_1 = df_abs_count_hit_target.loc[
    df_abs_count_hit_target['category'] == '1 ausencia',
    'Hit_target'
]

group_more_1 = df_abs_count_hit_target.loc[
    df_abs_count_hit_target['category'] == '>1 ausencias',
    'Hit_target'
]

u_statistic, p_value = stats.mannwhitneyu(
    group_1,
    group_more_1,
    alternative='two-sided'
)

print(f'U statistic: {u_statistic}')
print(f'p-value: {p_value}')

if p_value < 0.05:
    print('Reject H0: hay diferencias estadísticamente significativas entre los grupos.')
else:
    print('Fail to reject H0: no hay diferencias estadísticamente significativas entre los grupos.')

##### Son

In [ ]:
df_RRHH_ind_grouped['Son'].value_counts(normalize=True).sort_index()

In [ ]:
df_son_hit_target = df_RRHH_ind_grouped[['ID','Son','Hit_target']].copy()
df_son_hit_target

0 no tiene hijos y 1 sí que tiene

In [ ]:
df_son_hit_target['category'] = np.where(
    df_son_hit_target['Son'] == 0,
    '0',
    '1'
)

In [ ]:
df_son_hit_target['category'].value_counts()

In [ ]:
group_0 = df_son_hit_target.loc[
    df_son_hit_target['category'] == '0',
    'Hit_target'
]

group_1 = df_son_hit_target.loc[
    df_son_hit_target['category'] == '1',
    'Hit_target'
]

u_statistic, p_value = stats.mannwhitneyu(
    group_0,
    group_1,
    alternative='two-sided'
)

print(f'U statistic: {u_statistic}')
print(f'p-value: {p_value}')

if p_value < 0.05:
    print('Reject H0: hay diferencias estadísticamente significativas entre los grupos.')
else:
    print('Fail to reject H0: no hay diferencias estadísticamente significativas entre los grupos.')

##### Pet

In [ ]:
df_RRHH_ind_grouped['Pet'].value_counts(normalize=True).sort_index()

In [ ]:
df_pet_hit_target = df_RRHH_ind_grouped[['ID','Pet','Hit_target']].copy()
df_pet_hit_target

0 no tiene perros y 1 sí que tiene

In [ ]:
df_pet_hit_target['category'] = np.where(
    df_pet_hit_target['Pet'] == 0,
    '0',
    '1'
)

In [ ]:
df_son_hit_target['category'].value_counts()

In [ ]:
group_0 = df_pet_hit_target.loc[
    df_pet_hit_target['category'] == '0',
    'Hit_target'
]

group_1 = df_pet_hit_target.loc[
    df_pet_hit_target['category'] == '1',
    'Hit_target'
]

u_statistic, p_value = stats.mannwhitneyu(
    group_0,
    group_1,
    alternative='two-sided'
)

print(f'U statistic: {u_statistic}')
print(f'p-value: {p_value}')

if p_value < 0.05:
    print('Reject H0: hay diferencias estadísticamente significativas entre los grupos.')
else:
    print('Fail to reject H0: no hay diferencias estadísticamente significativas entre los grupos.')

##### Education

In [ ]:
df_RRHH_ind_grouped['Education'].value_counts(normalize=True).sort_index()

In [ ]:
df_education_hit_target = df_RRHH_ind_grouped[['ID','Education','Hit_target']].copy()
df_education_hit_target

0 tiene educación secundaria y 1 tiene estudios superiores

In [ ]:
df_education_hit_target['category'] = np.where(
    df_education_hit_target['Education'] == 1,
    '0',
    '1'
)

In [ ]:
df_education_hit_target['category'].value_counts()

In [ ]:
df_education_hit_target.groupby('category')['Hit_target'].agg(
    ['mean', 'min', 'max', 'median']
)

In [ ]:
group_0 = df_education_hit_target.loc[
    df_education_hit_target['category'] == '0',
    'Hit_target'
]

group_1 = df_education_hit_target.loc[
    df_education_hit_target['category'] == '1',
    'Hit_target'
]

u_statistic, p_value = stats.mannwhitneyu(
    group_0,
    group_1,
    alternative='two-sided'
)

print(f'U statistic: {u_statistic}')
print(f'p-value: {p_value}')

if p_value < 0.05:
    print('Reject H0: hay diferencias estadísticamente significativas entre los grupos.')
else:
    print('Fail to reject H0: no hay diferencias estadísticamente significativas entre los grupos.')

In [ ]:
df_education_hit_target.boxplot(
    column='Hit_target',
    by='category'
)

plt.title('Hit_target segons education')
plt.suptitle('')
plt.xlabel('Education')
plt.ylabel('Hit_target')
plt.show()

Noelia

In [ ]:
df_RRHH_ind_grouped['category_education'] = np.where(
    df_RRHH_ind_grouped['Education'] == 1,
    '0',
    '1'
)

In [ ]:
df_RRHH_ind_grouped.columns

In [ ]:
df_RRHH_ind_grouped.head(1)

In [ ]:
df_RRHH_ind_grouped.info()

In [ ]:
df_RRHH_ind_grouped['category_education'] = df_RRHH_ind_grouped['category_education'].astype(int)

In [ ]:
def analisis_mw_boxplot(df, variables_binarias, target='Hit_target'):
    """
    Realiza el test de Mann-Whitney U y genera visualizaciones de tipo Boxplot
    para comparar el desempeño entre grupos binarios.
    """
    resultados_mw = []

    fig, axes = plt.subplots(
        1,
        len(variables_binarias),
        figsize=(6 * len(variables_binarias), 6),
        sharey=True,
        facecolor='none'
    )

    if len(variables_binarias) == 1:
        axes = [axes]

    for i, var in enumerate(variables_binarias):

        grupo_0 = df[df[var] == 0][target].dropna()
        grupo_1 = df[df[var] == 1][target].dropna()

        stat, p_val = stats.mannwhitneyu(
            grupo_0,
            grupo_1,
            alternative='two-sided'
        )

        significativo = "SÍ" if p_val < 0.05 else "NO"

        resultados_mw.append({
            'Variable': var,
            'Estadístico U': stat,
            'P-Value': round(p_val, 4),
            'Diferencia Significativa': significativo
        })

        ax = axes[i]

        sns.boxplot(
            ax=ax,
            x=var,
            y=target,
            data=df,
            palette="Set2"
        )

        ax.set_facecolor('none')

        ax.set_title(f'Comparación de {target}\npor {var}')
        ax.set_xlabel(f'{var} (0=No, 1=Sí)')
        ax.set_ylabel(target if i == 0 else "")

    plt.tight_layout()
    plt.show()

    return pd.DataFrame(resultados_mw)

In [ ]:
vars_estudio = ['Social_drinker', 'Social_smoker', 'Disciplinary_failure','category_education']
df_stats_boxplot = analisis_mw_boxplot(df_RRHH_ind_grouped, vars_estudio)
print(df_stats_boxplot)

##### Disciplinary failure

In [ ]:
df_RRHH_ind_grouped['Disciplinary_failure'].value_counts()

In [ ]:
group_0 = df_RRHH_ind_grouped.loc[
    df_RRHH_ind_grouped['Disciplinary_failure'] == 0,
    'Hit_target'
].dropna()

group_1 = df_RRHH_ind_grouped.loc[
    df_RRHH_ind_grouped['Disciplinary_failure'] == 1,
    'Hit_target'
].dropna()

u_statistic, p_value = stats.mannwhitneyu(
    group_0,
    group_1,
    alternative='two-sided'
)

print(f'U statistic: {u_statistic}')
print(f'p-value: {p_value}')

if p_value < 0.05:
    print('Reject H0: hay diferencias estadísticamente significativas entre los grupos.')
else:
    print('Fail to reject H0: no hay diferencias estadísticamente significativas entre los grupos.')

In [ ]:
group_0.describe().T

In [ ]:
group_1.describe().T

#### Variables a categorizar

##### Absence count

In [ ]:
df_RRHH_ind_grouped['Absence_count'].value_counts(normalize=True).sort_index()

In [ ]:
df_abs_count_hit_target = df_RRHH_ind_grouped[['ID','Absence_count','Hit_target']].copy()
df_abs_count_hit_target

In [ ]:
df_abs_count_hit_target['category'] = np.where(
    df_abs_count_hit_target['Absence_count'] == 1,
    '1 ausencia',
    '>1 ausencias'
)

In [ ]:
df_abs_count_hit_target['category'].value_counts()

In [ ]:
group_1 = df_abs_count_hit_target.loc[
    df_abs_count_hit_target['category'] == '1 ausencia',
    'Hit_target'
]

group_more_1 = df_abs_count_hit_target.loc[
    df_abs_count_hit_target['category'] == '>1 ausencias',
    'Hit_target'
]

u_statistic, p_value = stats.mannwhitneyu(
    group_1,
    group_more_1,
    alternative='two-sided'
)

print(f'U statistic: {u_statistic}')
print(f'p-value: {p_value}')

if p_value < 0.05:
    print('Reject H0: hay diferencias estadísticamente significativas entre los grupos.')
else:
    print('Fail to reject H0: no hay diferencias estadísticamente significativas entre los grupos.')

##### Son

In [ ]:
df_RRHH_ind_grouped['Son'].value_counts(normalize=True).sort_index()

In [ ]:
df_son_hit_target = df_RRHH_ind_grouped[['ID','Son','Hit_target']].copy()
df_son_hit_target

0 no tiene hijos y 1 sí que tiene

In [ ]:
df_son_hit_target['category'] = np.where(
    df_son_hit_target['Son'] == 0,
    '0',
    '1'
)

In [ ]:
df_son_hit_target['category'].value_counts()

In [ ]:
group_0 = df_son_hit_target.loc[
    df_son_hit_target['category'] == '0',
    'Hit_target'
]

group_1 = df_son_hit_target.loc[
    df_son_hit_target['category'] == '1',
    'Hit_target'
]

u_statistic, p_value = stats.mannwhitneyu(
    group_0,
    group_1,
    alternative='two-sided'
)

print(f'U statistic: {u_statistic}')
print(f'p-value: {p_value}')

if p_value < 0.05:
    print('Reject H0: hay diferencias estadísticamente significativas entre los grupos.')
else:
    print('Fail to reject H0: no hay diferencias estadísticamente significativas entre los grupos.')

##### Pet

In [ ]:
df_RRHH_ind_grouped['Pet'].value_counts(normalize=True).sort_index()

In [ ]:
df_pet_hit_target = df_RRHH_ind_grouped[['ID','Pet','Hit_target']].copy()
df_pet_hit_target

0 no tiene perros y 1 sí que tiene

In [ ]:
df_pet_hit_target['category'] = np.where(
    df_pet_hit_target['Pet'] == 0,
    '0',
    '1'
)

In [ ]:
df_son_hit_target['category'].value_counts()

In [ ]:
group_0 = df_pet_hit_target.loc[
    df_pet_hit_target['category'] == '0',
    'Hit_target'
]

group_1 = df_pet_hit_target.loc[
    df_pet_hit_target['category'] == '1',
    'Hit_target'
]

u_statistic, p_value = stats.mannwhitneyu(
    group_0,
    group_1,
    alternative='two-sided'
)

print(f'U statistic: {u_statistic}')
print(f'p-value: {p_value}')

if p_value < 0.05:
    print('Reject H0: hay diferencias estadísticamente significativas entre los grupos.')
else:
    print('Fail to reject H0: no hay diferencias estadísticamente significativas entre los grupos.')

##### Education

In [ ]:
df_RRHH_ind_grouped['Education'].value_counts(normalize=True).sort_index()

In [ ]:
df_education_hit_target = df_RRHH_ind_grouped[['ID','Education','Hit_target']].copy()
df_education_hit_target

0 tiene educación secundaria y 1 tiene estudios superiores

In [ ]:
df_education_hit_target['category'] = np.where(
    df_education_hit_target['Education'] == 1,
    '0',
    '1'
)

In [ ]:
df_education_hit_target['category'].value_counts()

In [ ]:
df_education_hit_target.groupby('category')['Hit_target'].agg(
    ['mean', 'min', 'max', 'median']
)

In [ ]:
group_0 = df_education_hit_target.loc[
    df_education_hit_target['category'] == '0',
    'Hit_target'
]

group_1 = df_education_hit_target.loc[
    df_education_hit_target['category'] == '1',
    'Hit_target'
]

u_statistic, p_value = stats.mannwhitneyu(
    group_0,
    group_1,
    alternative='two-sided'
)

print(f'U statistic: {u_statistic}')
print(f'p-value: {p_value}')

if p_value < 0.05:
    print('Reject H0: hay diferencias estadísticamente significativas entre los grupos.')
else:
    print('Fail to reject H0: no hay diferencias estadísticamente significativas entre los grupos.')

In [ ]:
df_education_hit_target.boxplot(
    column='Hit_target',
    by='category'
)

plt.title('Hit_target segons education')
plt.suptitle('')
plt.xlabel('Education')
plt.ylabel('Hit_target')
plt.show()

Noelia

In [ ]:
df_RRHH_ind_grouped['category_education'] = np.where(
    df_RRHH_ind_grouped['Education'] == 1,
    '0',
    '1'
)

In [ ]:
df_RRHH_ind_grouped.columns

In [ ]:
df_RRHH_ind_grouped.head(1)

In [ ]:
df_RRHH_ind_grouped.info()

In [ ]:
df_RRHH_ind_grouped['category_education'] = df_RRHH_ind_grouped['category_education'].astype(int)

In [ ]:
def analisis_mw_boxplot(df, variables_binarias, target='Hit_target'):
    """
    Realiza el test de Mann-Whitney U y genera visualizaciones de tipo Boxplot
    para comparar el desempeño entre grupos binarios.
    """
    resultados_mw = []

    fig, axes = plt.subplots(
        1,
        len(variables_binarias),
        figsize=(6 * len(variables_binarias), 6),
        sharey=True,
        facecolor='none'
    )

    if len(variables_binarias) == 1:
        axes = [axes]

    for i, var in enumerate(variables_binarias):

        grupo_0 = df[df[var] == 0][target].dropna()
        grupo_1 = df[df[var] == 1][target].dropna()

        stat, p_val = stats.mannwhitneyu(
            grupo_0,
            grupo_1,
            alternative='two-sided'
        )

        significativo = "SÍ" if p_val < 0.05 else "NO"

        resultados_mw.append({
            'Variable': var,
            'Estadístico U': stat,
            'P-Value': round(p_val, 4),
            'Diferencia Significativa': significativo
        })

        ax = axes[i]

        sns.boxplot(
            ax=ax,
            x=var,
            y=target,
            data=df,
            palette="Set2"
        )

        ax.set_facecolor('none')

        ax.set_title(f'Comparación de {target}\npor {var}')
        ax.set_xlabel(f'{var} (0=No, 1=Sí)')
        ax.set_ylabel(target if i == 0 else "")

    plt.tight_layout()
    plt.show()

    return pd.DataFrame(resultados_mw)

In [ ]:
vars_estudio = ['Social_drinker', 'Social_smoker', 'Disciplinary_failure','category_education']
df_stats_boxplot = analisis_mw_boxplot(df_RRHH_ind_grouped, vars_estudio)
print(df_stats_boxplot)

### Kruskall-Wallis test  
[Kruskal Wallis test in Python - Medium](https://tnathu-ai.medium.com/kruskal-wallis-test-in-python-2899a3c87a8b)

In [ ]:
def kruskal_wallis_test(*groups, alpha=0.05, group_names=None):
    """
    Performs the Kruskal-Wallis test for 3 or more independent groups
    and prints descriptive statistics and test results.
    """

    if len(groups) < 3:
        raise ValueError("Kruskal-Wallis test requires at least 3 groups.")

    if group_names is None:
        group_names = [f'Group {i+1}' for i in range(len(groups))]

    if len(group_names) != len(groups):
        raise ValueError("group_names must have the same length as groups.")

    k = len(groups)
    degree_freedom = k - 1

    upper_tail_critical_value = stats.chi2.ppf(
        1 - alpha,
        df=degree_freedom
    )

    h_statistic, p_value = stats.kruskal(*groups)

    descriptors = []

    for name, group in zip(group_names, groups):
        group_series = pd.Series(group)

        q1 = group_series.quantile(0.25)
        q3 = group_series.quantile(0.75)

        descriptors.append({
            'group': name,
            'n': group_series.count(),
            'mean': group_series.mean(),
            'median': group_series.median(),
            'std': group_series.std(),
            'min': group_series.min(),
            'max': group_series.max(),
            'q1': q1,
            'q3': q3,
            'iqr': q3 - q1
        })

    descriptors_df = pd.DataFrame(descriptors)

    print('Descriptive statistics:')
    display(descriptors_df)

    print('\n---- Kruskal-Wallis test ----')
    print(f'Number of groups: {k}')
    print(f'Degrees of freedom: {degree_freedom}')
    print(f'Alpha: {alpha}')
    print(f'H statistic: {h_statistic}')
    print(f'P-value: {p_value}')
    print(f'Critical value X²U: {upper_tail_critical_value}')

    print('\n---- Interpretation using p-value ----')

    if p_value < alpha:
        print(
            f"The p-value is less than alpha {alpha}, so the result is significant.\n"
            "Reject the null hypothesis: at least one group is statistically different."
        )
    else:
        print(
            f"The p-value is larger than alpha {alpha}, so the result is not significant.\n"
            "Fail to reject the null hypothesis: there is no statistically significant difference between groups."
        )

    print('\n---- Interpretation using H statistic ----')

    if h_statistic < upper_tail_critical_value:
        print(
            "The Kruskal-Wallis H statistic is smaller than the critical value.\n"
            "Fail to reject the null hypothesis."
        )
    else:
        print(
            "The Kruskal-Wallis H statistic is larger than the critical value.\n"
            "Reject the null hypothesis."
        )

In [ ]:
df_RRHH_ind_grouped.columns

#### Variables directas

In [ ]:
vars_estudio = [
              'Education', 'Son','Pet'
              ]



##### Hit target Vs Education

In [ ]:
col_unique_values = sorted(df_RRHH_ind_grouped['Education'].unique().tolist())

# df_value = df_RRHH_ind_grouped[df_RRHH_ind_grouped['Education'] == value]

print(col_unique_values)

Crear grupos

In [ ]:
df_education_1 = df_RRHH_ind_grouped[df_RRHH_ind_grouped['Education'] == 1]['Hit_target']
df_education_2 = df_RRHH_ind_grouped[df_RRHH_ind_grouped['Education'] == 2]['Hit_target']
df_education_3 = df_RRHH_ind_grouped[df_RRHH_ind_grouped['Education'] == 3]['Hit_target']
df_education_4 = df_RRHH_ind_grouped[df_RRHH_ind_grouped['Education'] == 4]['Hit_target']

Comprovar grupo 1 (como ejemplo)

In [ ]:
df_education_1

Comprovar el tamaño de cada uno

In [ ]:
print(df_education_1.shape)
print(df_education_2.shape)
print(df_education_3.shape)
print(df_education_4.shape)


In [ ]:
kruskal_wallis_test(
    df_education_1,
    df_education_2,
    df_education_3,
    df_education_4,
    alpha=0.05,
    group_names=['Education 1', 'Education 2', 'Education 3', 'Education 4']
)

##### Hit target Vs Son

In [ ]:
col_unique_values = sorted(df_RRHH_ind_grouped['Son'].unique().tolist())

# df_value = df_RRHH_ind_grouped[df_RRHH_ind_grouped['Education'] == value]

print(col_unique_values)

Crear grupos

In [ ]:
df_son_0 = df_RRHH_ind_grouped[df_RRHH_ind_grouped['Son'] == 0]['Hit_target']
df_son_1 = df_RRHH_ind_grouped[df_RRHH_ind_grouped['Son'] == 1]['Hit_target']
df_son_2 = df_RRHH_ind_grouped[df_RRHH_ind_grouped['Son'] == 2]['Hit_target']
df_son_3 = df_RRHH_ind_grouped[df_RRHH_ind_grouped['Son'] == 3]['Hit_target']
df_son_4 = df_RRHH_ind_grouped[df_RRHH_ind_grouped['Son'] == 4]['Hit_target']

Comprovar grupo 1 (como ejemplo)

In [ ]:
df_son_0

Comprovar el tamaño de cada uno

In [ ]:
df_RRHH_ind_grouped['Son'].value_counts()

In [ ]:
print(df_son_0.shape)
print(df_son_1.shape)
print(df_son_2.shape)
print(df_son_3.shape)
print(df_son_4.shape)


In [ ]:
kruskal_wallis_test(
    df_son_0,
    df_son_1,
    df_son_2,
    df_son_3,
    df_son_4,
    alpha=0.05,
    group_names=['Son 0', 'Son 1', 'Son 2', 'Son 3', 'Son 4']
)

##### Hit target Vs Pet

In [ ]:
col_unique_values = sorted(df_RRHH_ind_grouped['Pet'].unique().tolist())

# df_value = df_RRHH_ind_grouped[df_RRHH_ind_grouped['Education'] == value]

print(col_unique_values)

Crear grupos

In [ ]:
df_pet_0 = df_RRHH_ind_grouped[df_RRHH_ind_grouped['Pet'] == 0]['Hit_target']
df_pet_1 = df_RRHH_ind_grouped[df_RRHH_ind_grouped['Pet'] == 1]['Hit_target']
df_pet_2 = df_RRHH_ind_grouped[df_RRHH_ind_grouped['Pet'] == 2]['Hit_target']
df_pet_4 = df_RRHH_ind_grouped[df_RRHH_ind_grouped['Pet'] == 4]['Hit_target']
df_pet_5 = df_RRHH_ind_grouped[df_RRHH_ind_grouped['Pet'] == 5]['Hit_target']
df_pet_8 = df_RRHH_ind_grouped[df_RRHH_ind_grouped['Pet'] == 8]['Hit_target']

Comprovar grupo 1 (como ejemplo)

In [ ]:
df_pet_0

Comprovar el tamaño de cada uno

In [ ]:
df_RRHH_ind_grouped['Pet'].value_counts().sort_index()

In [ ]:
print(df_pet_0.shape)
print(df_pet_1.shape)
print(df_pet_2.shape)
print(df_pet_4.shape)
print(df_pet_5.shape)
print(df_pet_8.shape)


In [ ]:
kruskal_wallis_test(
    df_pet_0,
    df_pet_1,
    df_pet_2,
    df_pet_4,
    df_pet_5,
    df_pet_8,
    alpha=0.05,
    group_names=['Pet 0', 'Pet 1', 'Pet 2', 'Pet 4', 'Pet 5', 'Pet 8']
)

#### Variables a categorizar

vars_categorizar = [
                    'Transportation_expense', 'Distance_Residence_Work', 'BMI_calculated',
                    'Service_time', 'Age','Work_load_Average_day','Absenteeism_hours', 'Absence_count'
                    ]

- Distance_Residence_Work: <=10 km, 11-20 km, 21-30 km, >30 km.
- Service_time: <=5 años, 6-10 años, 11-15 años, >15 años.
- Age: <=30, 31-40, 41-50, >50.
- Transportation_expense: podríamos segmentarla por cuantiles para no inventar cortes arbitrarios.

##### Distance_Residence_Work

In [ ]:
df_distance_hit_target = df_RRHH_ind_grouped[['ID','Distance_Residence_Work','Hit_target']].copy()
df_distance_hit_target

def category_distance(valor):
    if valor <= 10:
        return '<= 10'
    elif valor < 20:
        return '11 - 20'
    elif valor < 30:
        return '21 - 30'
    else:
        return '>30'


df_distance_hit_target['category'] = (
    df_RRHH_ind_grouped['Distance_Residence_Work']
    .apply(category_distance)
)

In [ ]:
df_distance_hit_target['category'] = pd.cut(
    df_distance_hit_target['Distance_Residence_Work'],
    bins=[0, 10, 20, 30, float('inf')],
    labels=['<= 10', '11 - 20', '21 - 30', '>30'],
    include_lowest=True,
    right=True
)

df_distance_hit_target

In [ ]:
df_10 = df_distance_hit_target[
    df_distance_hit_target['category'] == '<= 10'
].copy()

df_11_20 = df_distance_hit_target[
    df_distance_hit_target['category'] == '11 - 20'
].copy()

df_21_30 = df_distance_hit_target[
    df_distance_hit_target['category'] == '21 - 30'
].copy()

df_30_plus = df_distance_hit_target[
    df_distance_hit_target['category'] == '>30'
].copy()

In [ ]:
kruskal_wallis_test(
    df_10['Hit_target'],
    df_11_20['Hit_target'],
    df_21_30['Hit_target'],
    df_30_plus['Hit_target'],
    alpha=0.05,
    group_names=['<= 10', '11 - 20', '21 - 30', '>30']
)

##### Service_time  
<=5 años, 6-10 años, 11-15 años, >15 años.

In [ ]:
df_service_hit_target = df_RRHH_ind_grouped[['ID','Service_time','Hit_target']].copy()
df_service_hit_target

In [ ]:
df_service_hit_target['category'] = pd.cut(
    df_service_hit_target['Service_time'],
    bins=[0, 5, 10, 15, float('inf')],
    labels=['<= 5', '6 - 10', '11 - 15', '>15'],
    include_lowest=True,
    right=True
)

df_service_hit_target

In [ ]:
df_service_5 = df_service_hit_target[
    df_service_hit_target['category'] == '<= 5'
].copy()

df_service_6_10 = df_service_hit_target[
    df_service_hit_target['category'] == '6 - 10'
].copy()

df_service_11_15 = df_service_hit_target[
    df_service_hit_target['category'] == '11 - 15'
].copy()

df_service_15_plus = df_service_hit_target[
    df_service_hit_target['category'] == '>15'
].copy()

In [ ]:
kruskal_wallis_test(
    df_service_5['Hit_target'],
    df_service_6_10['Hit_target'],
    df_service_11_15['Hit_target'],
    df_service_15_plus['Hit_target'],
    alpha=0.05,
    group_names=['<= 5', '6 - 10', '11 - 15', '>15']
)

##### Age

In [ ]:
df_age_hit_target = df_RRHH_ind_grouped[['ID','Age','Hit_target']].copy()
df_age_hit_target

In [ ]:
df_age_hit_target['category'] = pd.cut(
    df_age_hit_target['Age'],
    bins=[0, 30, 40, 50, float('inf')],
    labels=['<= 30', '31 - 40', '41 - 50', '>50'],
    include_lowest=True,
    right=True
)

df_age_hit_target

In [ ]:
df_age_30 = df_age_hit_target[
    df_age_hit_target['category'] == '<= 30'
].copy()

df_age_31_40 = df_age_hit_target[
    df_age_hit_target['category'] == '31 - 40'
].copy()

df_age_41_50 = df_age_hit_target[
    df_age_hit_target['category'] == '41 - 50'
].copy()

df_age_50_plus = df_age_hit_target[
    df_age_hit_target['category'] == '>50'
].copy()

In [ ]:
kruskal_wallis_test(
    df_age_30['Hit_target'],
    df_age_31_40['Hit_target'],
    df_age_41_50['Hit_target'],
    df_age_50_plus['Hit_target'],
    alpha=0.05,
    group_names=['<= 30', '31 - 40', '41 - 50', '>50']
)

##### Transportation_Expense

In [ ]:
df_RRHH_ind_grouped['Transportation_expense'].value_counts(normalize=True).sort_index()

In [ ]:
df_RRHH_ind_grouped['Transportation_expense'].quantile([0.25, 0.50, 0.75])

In [ ]:
df_transport_hit_target = df_RRHH_ind_grouped[['ID','Transportation_expense','Hit_target']].copy()
df_transport_hit_target

In [ ]:
df_transport_hit_target['category'] = pd.qcut(
    df_transport_hit_target['Transportation_expense'],
    q=4,
    labels=['Q1 bajo', 'Q2 medio-bajo', 'Q3 medio-alto', 'Q4 alto']
)

In [ ]:
df_transport_hit_target.groupby('category')['Transportation_expense'].agg(
    n='count',
    min='min',
    max='max'
)

In [ ]:
transport_categories = ['Q1 bajo', 'Q2 medio-bajo', 'Q3 medio-alto', 'Q4 alto']

transport_groups = [
    df_transport_hit_target.loc[
        df_transport_hit_target['category'] == category,
        'Hit_target'
    ]
    for category in transport_categories
]

kruskal_wallis_test(
    *transport_groups,
    alpha=0.05,
    group_names=transport_categories
)

##### Absentism hours

In [ ]:
df_RRHH_ind_grouped['Absenteeism_hours'].value_counts(normalize=True).sort_index()

In [ ]:
plt.hist(df_RRHH_ind_grouped['Absenteeism_hours'], bins=4)
plt.show()

In [ ]:
df_abs_hours_hit_target = df_RRHH_ind_grouped[['ID','Absenteeism_hours','Hit_target']].copy()
df_abs_hours_hit_target

In [ ]:
df_abs_hours_hit_target['category'] = pd.cut(
    df_abs_hours_hit_target['Absenteeism_hours'],
    bins=[0, 8, 40, float('inf')],
    labels=['<= 8h', '9 - 40h', '>40h'],
    include_lowest=True,
    right=True
)

- <= 8h       hasta una jornada laboral
- 9 - 40h     entre más de 1 día y hasta una semana laboral
- > 40   ausencias acumuladas medias-altas

In [ ]:
df_abs_hours_hit_target.groupby('category', observed=True)['Absenteeism_hours'].agg(
    n='count',
    min='min',
    max='max'
)

In [ ]:
abs_hours_categories = ['<= 8h', '9 - 40h', '>40h']

abs_hours_groups = [
    df_abs_hours_hit_target.loc[
        df_abs_hours_hit_target['category'] == category,
        'Hit_target'
    ]
    for category in abs_hours_categories
]

kruskal_wallis_test(
    *abs_hours_groups,
    alpha=0.05,
    group_names=abs_hours_categories
)

**Seasons y Months no se puede hacer así porque no son observaciones independientes???**

## Rendiment mitjà per motiu d'absència

In [ ]:
# Rendiment mitjà per motiu d'absència
plt.figure(figsize=(14, 7))
reason_order = df_RRHH.groupby('Reason_absence_name')['Hit_target'].mean().sort_values(ascending=False).index

sns.barplot(data=df_RRHH, x='Reason_absence_name', y='Hit_target', 
            order=reason_order, palette='viridis')
plt.title('Rendiment mitjà (Hit_target) per motiu d\'absència')
plt.xlabel('Motiu d\'absència')
plt.ylabel('Hit_target (%)')
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.tight_layout()
plt.show()

# Visualizaciones presentación

## Violinplot

In [ ]:
# from analisis_absentismo:
pd.set_option('display.max_columns', 100)
sns.set_style('whitegrid')

COLOR_PRINCIPAL = 'teal'
COLOR_SECUNDARIO = 'steelblue'
COLOR_ALERTA = 'indianred'

#Añadimos color neutro: gris
COLOR_NEUTRO = '#bdc3c7'

In [ ]:
plt.figure(figsize=(7, 5), facecolor='none')

ax = (df_RRHH_ind_grouped['Hit_target']
      .value_counts(normalize=True)
      .sort_index()
      .mul(100)
      .plot(kind='bar', color=COLOR_PRINCIPAL))

ax.set_facecolor('none')

plt.ylabel('Recuento individuos (%)')
plt.xlabel('Desempeño (mediana)')
plt.title('Distribución del desempeño mediano por individuo')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5), facecolor='none')

hit_target_distribution = (
    df_RRHH_ind_grouped['Hit_target']
    .value_counts(normalize=True)
    .mul(100)
    .reindex(range(81, 101), fill_value=0)
)

ax = hit_target_distribution.plot(
    kind='bar',
    color=COLOR_PRINCIPAL
)

ax.set_facecolor('none')

plt.ylabel('Individuos (%)')
plt.xlabel('Desempeño mediano')
plt.title('Distribución del desempeño mediano por individuo')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(9, 5), facecolor='none')

hit_target_distribution = (
    df_RRHH_ind_grouped['Hit_target']
    .value_counts(normalize=True)
    .mul(100)
    .reindex(np.arange(81, 100.5 + 0.5, 0.5), fill_value=0)
)

ax = hit_target_distribution.plot(
    kind='bar',
    color=COLOR_PRINCIPAL,
    width=1.0
)

ax.set_facecolor('none')
ax.margins(x=0)

ax.set_xticklabels(
    [f'{x:g}' for x in hit_target_distribution.index],
    rotation=45
)

plt.ylabel('Individuos (%)')
plt.xlabel('Desempeño mediano')
plt.title('Distribución del desempeño mediano por individuo')
plt.tight_layout()
plt.show()

In [ ]:
df_education_hit_target = df_education_hit_target.copy()

df_education_hit_target['category_label'] = (
    df_education_hit_target['category']
    .astype(int)
    .map({
        0: 'Secundaria',
        1: 'Estudios superiores'
    })
)

In [ ]:
plt.figure(figsize=(7, 5), facecolor='none')

order = ['Secundaria', 'Estudios superiores']

ax = sns.violinplot(
    data=df_education_hit_target,
    x='category_label',
    y='Hit_target',
    order=order,
    palette=[COLOR_PRINCIPAL, COLOR_SECUNDARIO],
    inner='box'
)

# Transparency of the violin fill
for violin in ax.collections:
    violin.set_alpha(0.6)

sns.stripplot(
    data=df_education_hit_target,
    x='category_label',
    y='Hit_target',
    order=order,
    color='black',
    alpha=0.45,
    jitter=True
)

ax.set_facecolor('none')

plt.xlabel('')
plt.ylabel('Desempeño (%)')
plt.title('Desempeño según nivel educativo (segmentado)')

plt.tight_layout()
plt.show()

## Boxplot

In [ ]:
plt.figure(figsize=(7, 5), facecolor='none')

order = ['Secundaria', 'Estudios superiores']

ax = sns.boxplot(
    data=df_education_hit_target,
    x='category_label',
    y='Hit_target',
    order=order,
    palette=[COLOR_PRINCIPAL, COLOR_SECUNDARIO],
    showfliers=False
)

for patch in ax.patches:
    patch.set_alpha(0.6)

sns.stripplot(
    data=df_education_hit_target,
    x='category_label',
    y='Hit_target',
    order=order,
    color='black',
    alpha=0.45,
    jitter=True
)

ax.set_facecolor('none')

ax.set_yticks([80, 85, 90, 95, 100])
ax.set_ylim(78, 102)

plt.xlabel('')
plt.ylabel('Desempeño (%)')
plt.title('Desempeño según nivel educativo (categorizado)')

plt.tight_layout()
plt.show()

# df individuo (df_RRHH_ind_grouped)
**df_RRHH_ind_grouped** con:
- Variables sociodemográficas de cada individuo agrupadas
- Mediana de work_load_average_day y de hit target
- Conteo del número de ausencias (count de las solicitudes que tienen más de 0 horas)
- Sma de horas de absentismo.

In [ ]:
df_RRHH.columns

In [ ]:
df_RRHH_ind = df_RRHH[[
        'ID', 'Transportation_expense', 'Distance_Residence_Work',
        'Service_time', 'Age', 'Work_load_Average_day', 'Hit_target',
        'Disciplinary_failure', 'Education', 'Education_name', 'Son',
        'Social_drinker', 'Social_smoker', 'Pet', 'Weight', 'Height',
        'BMI_calculated', 'Absenteeism_hours',
        ]].copy()

In [ ]:
df_RRHH_ind.shape

In [ ]:
df_RRHH_ind.sample(5)

In [ ]:
cols_first = [
    'Transportation_expense', 'Distance_Residence_Work',
    'Service_time', 'Age',
    'Disciplinary_failure', 'Education', 'Education_name', 'Son',
    'Social_drinker', 'Social_smoker', 'Pet', 'Weight', 'Height',
    'BMI_calculated'
]

df_RRHH_ind_grouped = (
    df_RRHH_ind
    .groupby('ID')
    .agg(
        **{col: (col, 'first') for col in cols_first},
        Work_load_Average_day=('Work_load_Average_day', 'median'),
        Hit_target=('Hit_target', 'median'),
        Absenteeism_hours=('Absenteeism_hours', 'sum'),
        Absence_count=('Absenteeism_hours', lambda x: (x > 0).sum())
    )
    .reset_index()
)

df_RRHH_ind_grouped[['Work_load_Average_day', 'Hit_target']] = (
    df_RRHH_ind_grouped[['Work_load_Average_day', 'Hit_target']]
    .round(3)
)

In [ ]:
df_RRHH_ind_grouped

In [ ]:
df_RRHH_ind_grouped.columns

Listas de columnas numéricas y categóricas separadas para el estudio de individuo (_ID)

In [ ]:
num_ID = ['ID','Transportation_expense', 'Distance_Residence_Work', 'Service_time',
       'Age', 'Pet', 'Son', 'Weight', 'Height', 'BMI_calculated','Work_load_Average_day',
       'Hit_target', 'Absenteeism_hours', 'Absence_count']

cat_ID = ['ID','Disciplinary_failure', 'Education',
       'Social_drinker', 'Social_smoker']

## df ausencias sin variables sociodemográficas (df_RRHH_abs)
**df_RRHH_abs**  
Variables asociadas directamente con la solicitud de ausencia

In [ ]:
df_RRHH.columns

In [ ]:
col_absence = [
    'ID', 'Reason_absence', 'Reason_absence_name', 'Month_absence',
    'Month_absence_name', 'Day_week', 'Day_week_name', 'Seasons',
    'Seasons_name', 'Work_load_Average_day', 'Hit_target', 'Absenteeism_hours'
]

In [ ]:
df_RRHH_abs = df_RRHH[col_absence].copy()

df_RRHH_abs